# CIFAR-10 cross-victim RL pilot on Apple Silicon

This is a thin, package-backed research notebook. It trains a recurrent PPO attacker only against frozen residual and depthwise CNN source victims, then freezes the policy and evaluates transfer to a held-out patch transformer. The bounded Mac profile is a feasibility pilot and is explicitly **not research-valid**.

The default path only analyzes the committed aggregate snapshot. Set `RUN_TRAINING = True` to launch the full MPS/CPU run; raw traces and checkpoints remain under the ignored `output/` directory.

In [ ]:
import json
from pathlib import Path
import subprocess
import sys

def find_repo_root(start: Path) -> Path:
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'pyproject.toml').is_file() and (candidate / 'rl_transfer').is_dir():
            return candidate
    raise RuntimeError('Run this notebook from inside the deeppoly-cpp repository')

REPO_ROOT = find_repo_root(Path.cwd())
CONFIG_PATH = REPO_ROOT / 'configs/rl_transfer/cifar10_m4_pilot.json'
SNAPSHOT_PATH = REPO_ROOT / 'docs/research/cifar10_m4_pilot_results.json'
RUN_TRAINING = False
DEVICE = 'auto'
print(f'Repository: {REPO_ROOT}')

## Optional training

Install the project extras once with `uv pip install -e '.[vision,analysis,notebook,test]'`. The runner uses MPS when available, writes SHA-256-verified checkpoints after every policy block, and resumes only when the config, split, and package-code fingerprint match.

In [ ]:
if RUN_TRAINING:
    subprocess.run(
        [sys.executable, '-m', 'rl_transfer.cifar_cli', '--config', str(CONFIG_PATH), '--device', DEVICE],
        cwd=REPO_ROOT,
        check=True,
    )
else:
    print('Training disabled; analyzing existing artifacts or the committed snapshot.')

In [ ]:
output_root = REPO_ROOT / 'output/rl_transfer/cifar10_m4'
completed = []
for path in output_root.glob('*/manifest.json'):
    candidate = json.loads(path.read_text())
    if candidate.get('status') == 'complete' and candidate.get('name') == 'cifar10-m4-pilot':
        completed.append((path.stat().st_mtime, path, candidate))
if completed:
    _, evidence_path, evidence = max(completed)
else:
    evidence_path = SNAPSHOT_PATH
    evidence = json.loads(SNAPSHOT_PATH.read_text())
print(f'Evidence: {evidence_path.relative_to(REPO_ROOT)}')
print(f"Fingerprint: {evidence['fingerprint']}")
print(f"Research-valid: {evidence['research_valid']}")

## Data and victim-quality gates

Victim fitting, policy training, source validation, and outer target testing use deterministic, class-balanced splits. The transformer family is excluded from policy training.

In [ ]:
thresholds = evidence['victim_accuracy_gate']['thresholds']
print(f"{'family':<18} {'validation':>12} {'threshold':>12} {'pass':>8}")
for family, metrics in evidence['victims'].items():
    accuracy = metrics['source_validation_accuracy']
    threshold = thresholds[family]
    print(f"{family:<18} {accuracy:>12.3f} {threshold:>12.3f} {str(accuracy >= threshold):>8}")
print(f"Overall gate: {evidence['victim_accuracy_gate']['passed']}")
print(f"Held-out transformer test accuracy: {evidence['target_test_accuracy']:.3f}")

## Frozen cross-victim evaluation

All methods use the same clean-correct denominator and the same total target-query budget. `frozen=True` means the policy digest was identical before and after every deployment episode.

In [ ]:
evaluation = evidence['evaluation']
print(f"{'method':<38} {'success':>10} {'ASR@25':>10} {'AUC':>10} {'entropy':>10} {'frozen':>9}")
for method, metrics in evaluation.items():
    success = f"{metrics['successes']}/{metrics['eligible']}"
    print(f"{method:<38} {success:>10} {metrics['asr_at_budgets']['25']:>10.3f} {metrics['asr_query_auc']:>10.3f} {metrics['normalized_action_entropy']:>10.3f} {str(metrics['frozen']):>9}")

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    print("Install the analysis extra to render plots: uv pip install -e '.[analysis]'")
else:
    budgets = [0, 5, 10, 25]
    for method, metrics in evaluation.items():
        curve = metrics['asr_at_budgets']
        plt.plot(budgets, [curve[str(budget)] for budget in budgets], marker='o', label=method)
    plt.xlabel('Total target-query budget')
    plt.ylabel('Attack success rate (clean-correct denominator)')
    plt.title('Frozen transfer to held-out patch transformer')
    plt.legend()
    plt.grid(alpha=0.25)
    plt.show()

## Go/no-go decision

This bounded run passes the victim-quality gate, but the learned attacker must beat both fixed and random controls before scaling to multiple seeds or larger datasets. The result below is therefore a diagnostic decision, not a paper claim.

In [ ]:
learned = evaluation['groupdro_recurrent_ppo_stochastic']['asr_query_auc']
random_auc = evaluation['random_action']['asr_query_auc']
fixed_auc = evaluation['fixed_action']['asr_query_auc']
go = evidence['victim_accuracy_gate']['passed'] and learned > max(random_auc, fixed_auc)
decision = 'GO: repeat across seeds' if go else 'NO-GO: improve the policy/attack formulation before scaling'
print(decision)
print(f'learned AUC={learned:.4f}; random AUC={random_auc:.4f}; fixed AUC={fixed_auc:.4f}')
print(f"Runtime: {evidence['elapsed_seconds'] / 60:.1f} minutes on {evidence['device']['resolved']}")
print(f"Determinism note: {evidence['runtime']['determinism']}")